# Discover — 公開データを1行で探す

```python
results = search(text=KEYWORD, area=AREA,
                 providers=[ProviderId.SEARCH_CKAN_JP], limit=3)
```

下の一覧が実行結果です。**KEYWORD / AREA** を変えると別のテーマや地域を探せます。
検索ではデータ本体をダウンロードしません。気になった配布物は[02 Open](02_open_vector.ipynb)で地図にできます。
保存出力は2026-10-20に実セルをPythonプロセスで実行した観測例です。Colab/Jupyterの通し実行は未確認です。

## 準備

固定コミットのRhinestoneを導入します。GIS RuntimeはこのNotebookでは不要です。


In [1]:
import os
import subprocess
import sys

if os.environ.get("RHINESTONE_SHOWCASE_LOCAL") != "1":
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "rhinestone @ git+https://github.com/u-kitazawa/rhinestone.git@9bae025b08357d23e5f5339c25a777a1445da176",
        ]
    )
else:
    print("Local smoke: explicit checkout and caller-owned dependencies")

Local smoke: explicit checkout and caller-owned dependencies


In [2]:
from rhinestone import ProviderId, search

KEYWORD = "公園"
AREA = "那須塩原市"

In [3]:
results = None
results = search(
    text=KEYWORD, area=AREA, providers=[ProviderId.SEARCH_CKAN_JP], limit=3
)
for diagnostic in results.diagnostics:
    print(
        "Diagnostic:", diagnostic.source_id, diagnostic.reason, diagnostic.failure_type
    )
for provider_id in results.keys():
    print("Provider:", provider_id, "Results:", len(results[provider_id]))
    for candidate in results[provider_id]:
        print(
            candidate.title,
            "|",
            candidate.format,
            "|",
            candidate.metadata.publisher or "作成者不明",
        )
        print("  ", candidate.uri)
if not results:
    if results.diagnostics:
        raise ValueError("Providerの診断を確認してください。データ不在とは限りません。")
    raise ValueError("検索0件です。KEYWORD / AREAを変更してください。")

Provider: search-ckan-jp Results: 3
【那須塩原市】都市公園 | csv | 那須塩原市　企画部　企画政策課
   https://nasu-ckan.s3.amazonaws.com/ckan/resources/d4500d50-5649-4cb8-b55f-f0b45981abaf/40905.csv
【那須塩原市】公園 スポーツ振興 | csv | 那須塩原市 教育部 スポーツ振興課
   https://nasu-ckan.s3.amazonaws.com/ckan/resources/94cd5412-4f52-41c6-82a1-75092d15af4c/011201.csv
【那須塩原市】公園 都市整備 | csv | 那須塩原市　建設部　保全管理課
   https://nasu-ckan.s3.amazonaws.com/ckan/resources/0ea3bc07-b93b-4e22-8081-517845bd2eab/011201.csv


## 次に開く

一覧のタイトル・形式・作成者・URIを確認し、[02](02_open_vector.ipynb)のSELECTED_URIで明示選択します。
地域指定はmetadata検索の条件で、ファイル中の地物の切り出しではありません。広域データや地域に言及しただけの候補も残り得ます。
`limit=3`はProviderごとのResource数です。配布物のサイズや検索APIの総要求数の上限ではありません。
検索成功だけで実ファイルの可用性・driver対応は保証されません。

## 複数Providerへ広げる

必要なら上の検索セルのprovidersを次に変えられます。

```python
providers=[ProviderId.SEARCH_CKAN_JP, ProviderId.GSI]
```

結果はProviderごとのグループと構成順です。横断の関連度ランキングではありません。
提供元が対応しない条件や障害はdiagnosticsを確認してください。提供元によって0件になることもあります。

発見側と配布側の詳細な来歴は[発展編](advanced/discovery_lineage.ipynb)で確認できます。
出典・利用条件は配布物ごとに確認し、Openする前に[データ利用の説明](README.md#データと再現性)を読んでください。
